##**Rag Implemenation**

In [ ]:
!pip install -qU langchain-community langchain-google-genai langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.

**Importing All Necessary Libraries**

In [ ]:
from langchain_community.document_loaders import TextLoader # it loads the .txt files
from langchain_text_splitters import RecursiveCharacterTextSplitter # Create chuncks of my data
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma # This will help us to create vector store
from langchain_core.prompts import ChatPromptTemplate # this will help us to create system prompt for llm
from langchain_core.output_parsers import StrOutputParser # this will help us to parse the output of model in the string format
from langchain_core.runnables import RunnablePassthrough # it allow us to create a runnable chain

In [ ]:
import os

In [ ]:
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"
# Go to AI Studion ---> Go for Get API KEY ---> Create a project (If not created) --> Create and copy api key
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

**We will be creating Module by module for Rag**

###**Load and splitting of Docs**

In [ ]:
def load_and_split(filepath):
  loader = TextLoader(filepath)
  docs = loader.load() # It allow us to load the data from my file path
  # lazy load --> It is used when we have Large Dataset

  print(f'Splitting Data into Chunks...')
  splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
  splits = splitter.split_documents(docs)
  print(f'Split {len(splits)} Chunks')
  return splits

**Creating my Rag Chain**

In [ ]:
def create_rag_chain(splits):
  # we need to embed the data into vectors
  # taskType - 'RETRIEVAL_DOCUMENT' for embedding the docs
  embeddings = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001', task_type='RETRIEVAL_DOCUMENT')
  # we will pass embeddings to my vector store
  vectorstore = Chroma.from_documents(documents=splits,embedding=embeddings)
  retriver = vectorstore.as_retriever() # This is going to find context from my database

  # since our retriver is ready, We will create our LLM
  llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=1)
  # temperature (0-1): It allows us to increase or decrease the creativeness of the model
  template = """Answer the question based only on the following context: {context}
                Question: {question}

                Helpful Answer:"""

  prompt = ChatPromptTemplate.from_template(template) # System Prompt

  # We will create a function that will join the retrived chunks into one document then we will pass that doc as context
  def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)


  # Rag Chain
  # retriver --> Take question --> Embed it ---> Retrive from vector store ---> format_docs ---> It will be passed as context
  chain = (
      {'context': retriver | format_docs, 'question': RunnablePassthrough() }
      | prompt
      | llm
      | StrOutputParser()
  )

  return chain

**We need to create a Data about something and put it into a txt file.**

In [ ]:
file_path = '/content/harrypotter.txt'

splits = load_and_split(file_path)

Splitting Data into Chunks...
Split 27 Chunks


In [ ]:
rag_chain = create_rag_chain(splits)

In [ ]:
message = input('Ask Question Related to harry Potter: ')

output = rag_chain.invoke(message)
print(output)

KeyboardInterrupt: Interrupted by user

**Bonus**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def calculateSquare(number):
  return number * number

**If you want to create custom layout you should use gr.Blocks**

In [ ]:
with gr.Blocks() as demo:
  gr.Markdown('## Simple Square Calculator')

  with gr.Row():
    num_input = gr.Number(label ='Enter a Number')
    ResultOutput = gr.Number(label = 'Result')

  # we will be creating now a submit button
  submitBtn = gr.Button('Caluclate Square')

  # what happesn when we click the button
  # it takes num inpput, run it through calculateSquare and puts the answe in result output var
  submitBtn.click(
      fn = calculateSquare,
      inputs = num_input,
      outputs = ResultOutput
  )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://407dbd5f31000838d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Now using Gradio for our Model**

In [ ]:
def respond(message, history):
  return rag_chain.invoke(message)

In [ ]:
demo = gr.ChatInterface(
    fn = respond,
    textbox = gr.Textbox(placeholder='Ask a question regarding Harry Potter..')
)

demo.launch(share=True, debug = True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5c22a57a921f6fb099.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py", line 3194, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5864, in generate_content
    response = self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4526, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1402, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1236,

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ab7e3c46f8ffc56fb0.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://407dbd5f31000838d9.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://95c37a64737ffed430.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://97a56688b99df73b17.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://940db70d4461a4858e.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://5c22a57a921f6fb099.gradio.live


https://miro.com/app/board/uXjVHLm-_0Q=/?share_link_id=161231357730